# 01 — Khám phá dữ liệu thực tế
Năm nộp đầu tiên và năm cập nhật là hai biến khác nhau. Số dòng, khoảng năm và nhóm ngành được tính từ tập đã tải.

In [ ]:
import sys
from pathlib import Path
root = Path.cwd()
if not (root / "src").is_dir():
    root = root.parent
sys.path.insert(0, str(root / "src"))
from spark_common import create_spark_session
from project_config import DATA_DIR, RESULTS_DIR, OUTPUT_DIR
from data_contract import add_submission_year, add_categories
from pyspark.sql import functions as F
spark = create_spark_session("ArXivEDA")
df = spark.read.parquet(str(DATA_DIR / "arxiv_data.parquet"))
df.printSchema()
print("Rows:", df.count())
df.show(5, truncate=60)


In [ ]:
dated = add_categories(add_submission_year(df))
dated.select("id", "year", "submission_year", "update_year", "primary_category", "main_group").show(20)
dated.groupBy("year").count().orderBy("year").show(100)
dated.groupBy("main_group").count().orderBy(F.desc("count")).show(20)
dated.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in ["id", "title", "abstract", "categories", "year"]]).show()
print("Duplicate ID groups:", df.groupBy("id").count().filter("count > 1").count())


In [ ]:
spark.stop()
